# OpenPKFlow Demo

A complete dissolution similarity workflow — from raw data to regulatory-style report.
No numpy, matplotlib, or pandas imports needed.

In [ ]:
!pip install openpkflow --quiet

## 1. Quick f1 / f2 from arrays

In [ ]:
from openpkflow.dissolution import f1, f2

reference = [20.0, 40.0, 60.0, 80.0, 90.0]
test      = [21.0, 39.0, 61.0, 79.0, 88.0]

print(f"f1 = {f1(reference, test):.2f}  (<=15 acceptable)")
print(f"f2 = {f2(reference, test):.2f}  (>=50 similar)")

## 2. Load a CSV, compare, and print a summary

In [ ]:
from openpkflow.dissolution import DissolutionStudy
from openpkflow.datasets import example_dissolution_path

study = DissolutionStudy.from_csv(example_dissolution_path())
result = study.compare(reference="reference", test="test")

print(result.summary())

## 3. Profile plot — one line

In [ ]:
result.plot(output_path="profile.png", show=True)

## 4. Bootstrap f2 — from the same study, no numpy required

In [ ]:
boot = study.bootstrap_compare("reference", "test", n_replicates=5000, confidence_level=0.90, seed=42)
print(boot.summary())

## 5. Compare all three example datasets

In [ ]:
from openpkflow.datasets import example_dissolution_path, example_similar_path, example_not_similar_path

datasets = {
    "Borderline similar": example_dissolution_path(),
    "Clearly similar":    example_similar_path(),
    "Not similar":        example_not_similar_path(),
}

print(f"{'Dataset':<22} {'f1':>6} {'f2':>6} {'Similar':>8}")
print("-" * 46)
for label, path in datasets.items():
    r = DissolutionStudy.from_csv(path).compare(reference="reference", test="test")
    print(f"{label:<22} {r.f1_value:>6.2f} {r.f2_value:>6.2f} {str(r.f2_value >= 50):>8}")

## 6. Regulatory f2 — FDA 85% rule applied automatically

In [ ]:
ref_late = [20.0, 40.0, 60.0, 80.0, 90.0, 95.0]
tst_late = [20.0, 40.0, 60.0, 80.0, 92.0, 96.0]

print(f"f2 all_points : {f2(ref_late, tst_late, method='all_points'):.2f}")
print(f"f2 regulatory : {f2(ref_late, tst_late, method='regulatory'):.2f}")
print("Regulatory mode trims the second timepoint where both profiles exceed 85%.")

## 7. Generate a full HTML report

In [ ]:
result.report("dissolution_report.html", format="html")
print("Saved dissolution_report.html — open in a browser to see the full regulatory-style report.")